# Matching luchtaanvallen Gaza en safezones

In deze notebook wordt bepaald wanneer en welke safezones op de dezelfde dag dat ze zijn aangekondigd, worden gebombardeerd.

In [4]:
import pandas as pd
import re

# Load datasets
attacks = pd.read_csv('acled_gaza_may23.csv')
leaflets = pd.read_csv('displacement_updated.csv')


In [5]:
# select dates from end of ceasefire to May 23

attacks['event_date'] = pd.to_datetime(attacks['event_date'])
attacks_cease = attacks.loc[attacks['event_date'].between('2025-03-18','2025-05-23', inclusive='both')]

leaflets['date'] = pd.to_datetime(leaflets['date'])
leaflets_cease = leaflets.loc[leaflets['date'].between('2025-03-18','2025-05-23', inclusive='both')]



In [6]:
# leaflets_cease.info()

# leaflets_cease.head()

In [7]:

# Filter rows where safezone is 'east', 'south', 'north', or 'west'
filtered_leaflets = leaflets_cease[leaflets_cease['safezone'].isin(['east', 'south', 'north', 'west'])]

filtered_leaflets

,date,x_source,x_text,x_text_translated,x_timestamp_utc3,evacuation_zone_prompt,facebook_source,facebook_timestamp_utc3,leaflet,leaflet_text_translated,...,link,map_idf,map_full,map_zoom,displacement_blocks,labeled_safe_blocks,evacuation_zone,safezone,area_sq_km_displacement,area_sq_km_labeled_safe
1,2025-05-21,https://x.com/AvichayAdraee/status/19252451965...,#عاجل ‼️ تحذير خطير الى كل سكان قطاع غزة المتو...,A serious warning to all residents of the Gaza...,20:39,"Gaza Strip, Ghaban, Al-Shimaa, Fadous, Al-Mans...",https://www.facebook.com/IDFarabicAvichayAdrae...,20:41,NaN,NaN,...,https://gazamaps.com/displacement/101,https://gazamaps.com/storage/displacement-maps...,https://gazamaps.com/storage/displacement-maps...,https://gazamaps.com/storage/displacement-maps...,"961, 962, 963, 964, 965, 966, 967, 968, 969, 9...",NaN,"Gaza Strip, Ghaban, Al-Shimaa, Fadous, Al-Mans...",south,13.26,0.0
23,2025-03-24,https://x.com/avichayadraee/status/19042676177...,#عاجل ‼️ إلى جميع سكان قطاع غزة المتواجدين في ...,To all residents of the Gaza Strip located in ...,23:22,"Gaza Strip, Jabalia area",https://www.facebook.com/IDFarabicAvichayAdrae...,23:23,NaN,NaN,...,https://gazamaps.com/displacement/80,https://gazamaps.com/storage/displacement-maps...,https://gazamaps.com/storage/displacement-maps...,https://gazamaps.com/storage/displacement-maps...,"754, 755, 756, 1771, 1772, 1782, 1784, 1785",NaN,"Gaza Strip, Jabalia area",south,3.44,0.0
24,2025-03-24,https://x.com/avichayadraee/status/19042362243...,#عاجل ‼️ إلى جميع سكان قطاع غزة المتواجدين في ...,To all residents of the Gaza Strip in the Beit...,21:17,"Beit Lahia, Beit Hanoun",NaN,NaN,NaN,NaN,...,https://gazamaps.com/displacement/79,https://gazamaps.com/storage/displacement-maps...,https://gazamaps.com/storage/displacement-maps...,https://gazamaps.com/storage/displacement-maps...,"577, 578, 1765, 1767, 1768, 1769, 1770, 1770, ...",NaN,"Beit Lahia, Beit Hanoun",west,4.66,0.0
26,2025-03-21,https://x.com/avichayadraee/status/19031056548...,#عاجل ‼️ إلى جميع سكان قطاع غزة المتواجدين في ...,To all residents of the Gaza Strip who are in ...,18:25,"Gaza Strip, Al-Salateen, Madinat Al-Awda, Al-K...",NaN,NaN,NaN,NaN,...,https://gazamaps.com/displacement/77,https://gazamaps.com/storage/displacement-maps...,https://gazamaps.com/storage/displacement-maps...,https://gazamaps.com/storage/displacement-maps...,"974, 978, 979, 980, 982, 1742, 1743",NaN,"Gaza Strip, Sultans, Madinat Al-Awda, Al-Karama",south,5.57,0.0
27,2025-03-20,https://x.com/avichayadraee/status/19027145933...,#عاجل ‼️ إلى جميع سكان قطاع غزة المتواجدين في ...,To all residents of the Gaza Strip located in ...,16:31,"Gaza Strip, Bani Suhaila",NaN,NaN,NaN,NaN,...,https://gazamaps.com/displacement/76,https://gazamaps.com/storage/displacement-maps...,https://gazamaps.com/storage/displacement-maps...,https://gazamaps.com/storage/displacement-maps...,"37.1, 37.2, 38, 39, 41, 42.1, 42.2, 42.3, 220....",NaN,"Gaza Strip, Bani Suhaila",west,9.99,0.0


In [8]:
# remove rows where safezone = 'Unknown'

remove_values = ['unknown', 'south', 'west', 'east', 'north']

# Filter rows where 'safezone' (lowercased) is NOT in remove_values
leaflets_cease = leaflets_cease[
    ~leaflets_cease['safezone'].str.lower().isin(remove_values)
].reset_index(drop=True)

In [9]:
attacks_cease.info()

leaflets_cease.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1917 entries, 0 to 1916
Data columns (total 31 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   event_id_cnty       1917 non-null   object        
 1   event_date          1917 non-null   datetime64[ns]
 2   year                1917 non-null   int64         
 3   time_precision      1917 non-null   int64         
 4   disorder_type       1917 non-null   object        
 5   event_type          1917 non-null   object        
 6   sub_event_type      1917 non-null   object        
 7   actor1              1917 non-null   object        
 8   assoc_actor_1       0 non-null      object        
 9   inter1              1917 non-null   object        
 10  actor2              1232 non-null   object        
 11  assoc_actor_2       381 non-null    object        
 12  inter2              1232 non-null   object        
 13  interaction         1917 non-null   object        
 1

In [10]:
# !python -m spacy download en_core_web_lg

In [11]:
attacks_cease.head(3)

,event_id_cnty,event_date,year,time_precision,disorder_type,event_type,sub_event_type,actor1,assoc_actor_1,inter1,...,location,latitude,longitude,geo_precision,source,source_scale,notes,fatalities,tags,timestamp
0,PSE74167,2025-05-23,2025,1,Political violence,Explosions/Remote violence,Air/drone strike,Military Forces of Israel (2022-),NaN,External/Other forces,...,Jabalya,31.5272,34.4835,2,Newpress; Palestine News and Information Agenc...,New media-National,"On 23 May 2025, Israeli warplanes targeted a g...",3,NaN,1748304023
1,PSE74182,2025-05-23,2025,1,Political violence,Explosions/Remote violence,Air/drone strike,Military Forces of Israel (2022-),NaN,External/Other forces,...,Gaza - Southern Remal,31.5168,34.4355,2,Quds News Network,National,"On 23 May 2025, Israeli warplanes targeted sev...",0,NaN,1748304023
2,PSE74194,2025-05-23,2025,1,Political violence,Explosions/Remote violence,Air/drone strike,Military Forces of Israel (2022-),NaN,External/Other forces,...,An Nusayrat,31.4486,34.3925,1,Palestine News and Information Agency; Quds Ne...,National,"On 23 May 2025, Israeli warplanes targeted the...",3,NaN,1748304023


In [12]:
# replacing location names in displacement to match the location names in acled data

# western Gaza city -> Gaza - Southern Remal

leaflets_cease['safezone'] = leaflets_cease['safezone'].str.replace(
    'western Gaza City', 'Gaza - Southern Remal', regex=False
)

# detect Wadi Gaza in acled data 'notes' column, add to admin3 column

def update_admin3(row):
    if isinstance(row['notes'], str) and 'wadi gaza' in row['notes'].lower():
        return 'wadi gaza'
    return row['admin3']

attacks_cease['admin3'] = attacks_cease.apply(update_admin3, axis=1)

# map all instances of mawasi in leaflets_cease to 'Al Mawasi'

leaflets_cease['safezone'] = leaflets_cease['safezone'].apply(
    lambda x: 'Al Mawasi' if isinstance(x, str) and 'mawasi' in x.lower() else x
)

# map all instances of mawasi in attacks_cease to 'Al Mawasi'
def normalize_al_mawasi(loc):
    if isinstance(loc, str) and re.match(r'(?i)^al\s*mawasi\s*(\([^)]+\))?$', loc.strip()):
        return 'Al Mawasi'
    return loc

attacks_cease['location'] = attacks_cease['location'].apply(normalize_al_mawasi)


/tmp/ipython-input-12-1283564979.py:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  attacks_cease['admin3'] = attacks_cease.apply(update_admin3, axis=1)
/tmp/ipython-input-12-1283564979.py:30: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  attacks_cease['location'] = attacks_cease['location'].apply(normalize_al_mawasi)


In [13]:

# Step 1: Merge leaflets with attacks on 'safezone' matching any of the three columns in attacks
# Create three separate merges and then concatenate or combine results

merge_admin2 = pd.merge(leaflets_cease, attacks_cease, left_on='safezone', right_on='admin2', how='left', suffixes=('', '_attack_admin2'))
merge_admin3 = pd.merge(leaflets_cease, attacks_cease, left_on='safezone', right_on='admin3', how='left', suffixes=('', '_attack_admin3'))
merge_location = pd.merge(leaflets_cease, attacks_cease, left_on='safezone', right_on='location', how='left', suffixes=('', '_attack_location'))

# Step 2: Combine the merges to find all matches
combined = pd.concat([merge_admin2, merge_admin3, merge_location])

# Step 3: Drop duplicates if necessary (depending on your data structure)
combined = combined.drop_duplicates(subset=['safezone', 'event_date'])

# Step 4: Filter to get only those safe zones which were bombed (i.e., have matching attacks)
bombed_safezones = combined[combined['event_date'].notna()]  # replace with the actual date column

# Step 5: Select relevant columns to show safe zone, bombing date, and attack details
result = bombed_safezones[['safezone', 'event_date', 'admin2', 'admin3', 'location']]

# result now contains safe zones that were bombed with date and related attack info


In [14]:

# Ensure date columns are datetime
leaflets_cease['date'] = pd.to_datetime(leaflets_cease['date'])
attacks_cease['event_date'] = pd.to_datetime(attacks_cease['event_date'])

# Function to match on same day and location
def match_same_day(leaflets, attacks, location_field):
    merged = pd.merge(
        leaflets, attacks,
        left_on='safezone', right_on=location_field,
        how='inner'
    )
    merged = merged[merged['date'] == merged['event_date']]
    return merged

# Perform match for each location field
same_day_admin2 = match_same_day(leaflets_cease, attacks_cease, 'admin2')
same_day_admin3 = match_same_day(leaflets_cease, attacks_cease, 'admin3')
same_day_location = match_same_day(leaflets_cease, attacks_cease, 'location')

# Combine all matches
same_day_attacks = pd.concat([same_day_admin2, same_day_admin3, same_day_location], ignore_index=True)

# Remove duplicates if necessary
same_day_attacks = same_day_attacks.drop_duplicates(subset=['safezone', 'event_date', 'date'])

# Result: attacks that occurred on the same day a leaflet mentioned the corresponding safezone
result = same_day_attacks[['safezone', 'event_date', 'admin2', 'admin3', 'location']]

/tmp/ipython-input-14-300303897.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  attacks_cease['event_date'] = pd.to_datetime(attacks_cease['event_date'])


In [15]:
result

,safezone,event_date,admin2,admin3,location
0,Gaza City,2025-05-13,Gaza City,NaN,Gaza - Ash Shati' Camp
9,Gaza City,2025-04-25,Gaza City,NaN,Gaza
16,Gaza City,2025-04-24,Gaza City,NaN,Gaza
24,Khan Yunis,2025-04-11,Khan Yunis,NaN,Khan Yunis
32,Gaza City,2025-04-02,Gaza City,NaN,Gaza - Shujaiyya
34,Gaza City,2025-04-01,Gaza City,NaN,Al Mughraqa
41,Al Mawasi,2025-05-19,Khan Yunis,NaN,Al Mawasi
43,Al Mawasi,2025-05-18,Khan Yunis,NaN,Al Mawasi
44,Al Mawasi,2025-04-13,Khan Yunis,NaN,Al Mawasi
45,Al Mawasi,2025-04-12,Khan Yunis,NaN,Al Mawasi
